# Foundation Model - TabPFN v3.5 Fine Tuned - layer A

In [ ]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import sys
scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))
from config import final_data_path, hf_token

In [2]:
!pip install -U tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 16.4 MB/s eta 0:00:00


In [ ]:
final_data_path = Path(final_data_path)
features_dir = final_data_path
results_dir = final_data_path / "modeling_results" / "fm_tuned"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    key = f"{name}_subsampled"
    df = pd.read_parquet(features_dir / f"{key}_features.parquet")
    datasets[key] = df
    print(f"{key}: {df.shape}")

layer_a_bts_subsampled: (6966559, 28)
layer_b_bts_meteostat_subsampled: (6966559, 32)
layer_c_bts_gdelt_subsampled: (6966559, 43)
layer_d_bts_meteostat_gdelt_subsampled: (6966559, 47)


In [4]:
def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

In [5]:
def get_train_val_test(key: str, val_size=0.15, random_state=42):
    df = datasets[key]
    train_full = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    train, val = train_test_split(
        train_full, test_size=val_size, stratify=train_full[target_col], random_state=random_state
    )

    X_train, y_train = train[feature_cols], train[target_col]
    X_val, y_val = val[feature_cols], val[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_val, y_val, X_test, y_test

In [6]:
beta=1.2

def tune_threshold(y_true, oob_proba, beta=beta):
    precision, recall, thresholds = precision_recall_curve(y_true, oob_proba)
    fbeta_scores = (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall + 1e-12)
    )
    best_idx = np.nanargmax(fbeta_scores[:-1])
    return thresholds[best_idx]

In [7]:
def predict_proba_chunked(clf, X_test, batch_size=50_000, desc="predicting batches"):
    n = len(X_test)
    proba = np.empty(n, dtype=np.float64)

    for start in tqdm(range(0, n, batch_size), desc=desc):
        end = min(start + batch_size, n)
        proba[start:end] = clf.predict_proba(X_test.iloc[start:end])[:, 1]

    return proba

In [ ]:
os.environ["TABPFN_MODEL_VERSION"] = "v3.5"
os.environ["TABPFN_TOKEN"] = hf_token

In [ ]:
from tabpfn.finetuning import FinetunedTabPFNClassifier
from tabpfn.constants import ModelVersion

In [ ]:
class HistoryLogger:
    """Captures per-epoch metrics from fit() and prints them live."""
    def __init__(self):
        self.epoch_records = []

    def setup(self, config): pass
    def log_step(self, metrics, step): pass
    def log_epoch(self, metrics, step):
        self.epoch_records.append({**metrics, "step": step})
        if "val/ROC AUC" in metrics:
            print(f"    epoch {int(metrics['train/epoch'])}: val_auc={metrics['val/ROC AUC']:.4f}", flush=True)
    def finish(self): pass


results = {}
layer_probas = {}

tabpfn_n_estimators = 2
tabpfn_subsample_samples = 20_000
predict_batch_size = 20_000
beta = 1.2
val_metric_key = "val/ROC AUC"

best_lr = 3e-05
best_epochs = 20
ft_n_estimators_finetune = tabpfn_n_estimators
ft_n_estimators_final = tabpfn_n_estimators
ft_ctx_query_samples = 10_000

layers_to_run = ["layer_a_bts",]

for name in tqdm(layers_to_run, desc="TabPFN (fine-tuned) layers"):
    key = f"{name}_subsampled"
    x_train, y_train, x_val, y_val, x_test, y_test = get_train_val_test(key)

    t0 = time.time()
    history_logger = HistoryLogger()

    clf = FinetunedTabPFNClassifier(
        model_version=ModelVersion.V3_5,
        device="cuda",
        epochs=best_epochs,
        learning_rate=best_lr,
        early_stopping=True,
        early_stopping_patience=8,
        eval_metric="roc_auc",
        experiment_logger=history_logger,
        n_estimators_finetune=ft_n_estimators_finetune,
        n_estimators_final_inference=ft_n_estimators_final,
        n_inference_subsample_samples=tabpfn_subsample_samples,
        n_finetune_ctx_plus_query_samples=ft_ctx_query_samples,
        random_state=42,
    )
    clf.fit(
        x_train, y_train,
        X_val=x_val, y_val=y_val,
        output_dir=results_dir / "finetune_checkpoints" / key,
    )
    fit_seconds = time.time() - t0

    val_history = [r for r in history_logger.epoch_records if val_metric_key in r]
    epochs_run = len(val_history)
    best_record = max(val_history, key=lambda r: r[val_metric_key]) if val_history else None
    best_epoch = int(best_record["train/epoch"]) if best_record else None
    best_val_auc = best_record[val_metric_key] if best_record else None

    val_proba = predict_proba_chunked(clf, x_val, batch_size=predict_batch_size, desc="predicting val batches")
    threshold = tune_threshold(y_val, val_proba, beta=beta)

    y_proba = predict_proba_chunked(clf, x_test, batch_size=predict_batch_size, desc="predicting test batches")
    y_pred = (y_proba >= threshold).astype(int)
    elapsed = time.time() - t0

    metrics = {
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "threshold": threshold,
        "learning_rate": best_lr,
        "epochs_cap": best_epochs,
        "epochs_run": epochs_run,
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
        "n_estimators_finetune": ft_n_estimators_finetune,
        "n_estimators_final_inference": ft_n_estimators_final,
        "zero_shot": False,
        "fit_seconds": fit_seconds,
        "train_seconds": elapsed,
    }
    results[key] = metrics

    with open(results_dir / f"{key}_finetuned_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)


    pd.DataFrame(val_history).to_csv(results_dir / f"{key}_finetuned_epoch_history.csv", index=False)

    np.save(results_dir / f"{key}_finetuned_y_proba.npy", y_proba)
    np.save(results_dir / f"{key}_finetuned_y_pred.npy", y_pred)

    layer_probas[key] = {
        "y_val": y_val,
        "val_proba": val_proba,
        "y_test": y_test,
        "test_proba": y_proba,
    }

    print(f"{key}: {metrics}")

    del clf, history_logger, x_train, y_train, x_val, x_test, y_pred
    gc.collect()
    torch.cuda.empty_cache()

TabPFN (fine-tuned) layers:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/tabpfn/finetuning/finetuned_base.py:877: UserWarning: Output dir present but no checkpoint file found for train_size=850000. Starting from default checkpoint and epoch 0
  get_checkpoint_path_and_epoch_from_output_dir(


tabpfn-v3.5-20260909.safetensors:   0%|          | 0.00/876M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

Finetuning Epoch 1/20:   0%|          | 0/85 [00:00<?, ?it/s]

...

Finetuning Epoch 20/20:   0%|          | 0/85 [00:00<?, ?it/s]

    epoch 19: val_auc=0.7106


predicting val batches:   0%|          | 0/8 [00:00<?, ?it/s]

predicting test batches:   0%|          | 0/299 [00:00<?, ?it/s]

layer_a_bts_subsampled: {'f1': 0.42337733639823655, 'roc_auc': np.float64(0.6738203206008924), 'pr_auc': np.float64(0.35710978743936833), 'mcc': np.float64(0.20951851176446973), 'precision': 0.31268114775215766, 'recall': 0.6554057889415782, 'threshold': np.float64(0.18437910079956055), 'learning_rate': 3e-05, 'epochs_cap': 20, 'epochs_run': 20, 'best_epoch': 17, 'best_val_auc': np.float64(0.7105712444796222), 'n_estimators_finetune': 2, 'n_estimators_final_inference': 2, 'zero_shot': False, 'fit_seconds': 25886.369951963425, 'train_seconds': 32192.347818136215}


In [ ]:
with open(results_dir / "layer_a_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

pd.DataFrame(results).T

,f1,roc_auc,pr_auc,mcc,precision,recall,threshold,learning_rate,epochs_cap,epochs_run,best_epoch,best_val_auc,n_estimators_finetune,n_estimators_final_inference,zero_shot,fit_seconds,train_seconds
layer_a_bts_subsampled,0.423377,0.67382,0.35711,0.209519,0.312681,0.655406,0.184379,0.00003,20,20,17,0.710571,2,2,False,25886.369952,32192.347818


In [ ]:
summary_df = pd.DataFrame(results).T
summary_df.index.name = "layer"
summary_df = summary_df.reset_index()

summary_df = summary_df.rename(columns={"roc_auc": "auc_roc", "pr_auc": "auc_pr", "train_seconds": "fit_time_s"})
summary_df = summary_df[[
    "layer", "f1", "precision", "recall", "auc_roc", "auc_pr", "mcc", "threshold",
    "learning_rate", "epochs_cap", "epochs_run", "best_epoch", "best_val_auc",
    "n_estimators_finetune", "n_estimators_final_inference", "fit_time_s",
]]

summary_df.to_csv(results_dir / "fm_subsampled_summary_layer_a.csv", index=False)
summary_df

,layer,f1,precision,recall,auc_roc,auc_pr,mcc,threshold,learning_rate,epochs_cap,epochs_run,best_epoch,best_val_auc,n_estimators_finetune,n_estimators_final_inference,fit_time_s
0,layer_a_bts_subsampled,0.423377,0.312681,0.655406,0.67382,0.35711,0.209519,0.184379,0.00003,20,20,17,0.710571,2,2,32192.347818
